# Dashboard #4 — Export Predictions to PostgreSQL

Loads the trained model and generates predicted recession probabilities across the full
1990–present range, then writes the result to a new table so Power BI can connect to it
directly (Power BI has no Python/sklearn runtime — inference happens here, once, not
inside the dashboard).

In [4]:
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()

user = os.getenv('DB_USER')
password = os.getenv('DB_PASSWORD')
host = os.getenv('DB_HOST')
port = os.getenv('DB_PORT')
dbname = os.getenv('DB_NAME')

engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{dbname}")

## Load trained model and feature data

Uses the final 4-feature model (`BAA10Y`, `CFNAI`, `ICSA`, `INDPRO`) persisted in
`04_model_training.ipynb`. Only the feature columns are needed here — no target/dropna
against `target_h6`, since predictions are generated for the full available range, including
the most recent months where a forward-looking target can't yet exist.

In [1]:
import joblib
import pandas as pd

pipe = joblib.load('../data/processed/recession_model_h6.pkl')
monthly_1990 = pd.read_csv('../data/processed/monthly_1990.csv', index_col=0, parse_dates=True)

feature_cols = ['BAA10Y', 'CFNAI', 'ICSA', 'INDPRO']
X_full = monthly_1990[feature_cols].dropna()

## Generate predictions

Applies the *already-fitted* pipeline (scaler + logistic regression, fit on the training
fold in the previous notebook) to the full feature history — not refit on the full dataset,
so these predictions are consistent with the reported evaluation metrics.

In [2]:
predicted_proba = pipe.predict_proba(X_full)[:, 1]

output = pd.DataFrame({
    'obs_date': X_full.index,
    'predicted_probability': predicted_proba
})

## Create the target table

Following the project's existing convention of explicit DDL scripts under `sql/`, rather than
letting pandas infer column types automatically.

In [5]:
from sqlalchemy import text

with open('../sql/create_fact_recession_probability.sql') as f:
    ddl = f.read()

with engine.begin() as conn:
    conn.execute(text(ddl))

## Write predictions to PostgreSQL

`if_exists='replace'` fully recreates the table on each run — appropriate here since the
model is deterministic and refit on the full history each time the pipeline runs; an
append-only approach would accumulate duplicate/stale rows across reruns.

In [6]:
output.to_sql(
    'fact_recession_probability',
    engine,
    if_exists='replace',
    index=False
)

439